# Decorators and Closures

Decorators are one of Python's most powerful features — they allow you to **modify or enhance** functions/methods without changing their source code.

### Prerequisites
- First-class functions (functions can be passed as arguments)
- Nested functions
- `*args` and `**kwargs`

> **Prerequisite**: Functions, *args/**kwargs, scope.

## First-Class Functions

In Python, functions are **first-class citizens**: they can be assigned to variables, passed as arguments, and returned from other functions.

In [ ]:
# Functions are first-class objects
def greet(name):
    return f"Hello, {name}!"

# Assign to variable
my_func = greet
print(f"Via variable: {my_func('Alice')}")

# Pass as argument
def call_twice(func, arg):
    return func(arg), func(arg)

result = call_twice(greet, "Bob")
print(f"Called twice: {result}")

# Return from function
def get_operation(op):
    def multiply(a, b):
        return a * b
    def add(a, b):
        return a + b
    
    if op == '+':
        return add
    elif op == '*':
        return multiply

operation = get_operation('+')
print(f"Operation (3 + 4): {operation(3, 4)}")

## Closures

A **closure** is a nested function that **remembers** the variables from its enclosing scope even after that scope has finished executing.

In [ ]:
# Closure example
def make_multiplier(factor):
    """Return a function that multiplies by factor."""
    def multiplier(x):
        return x * factor  # 'factor' is captured (closure)
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)

print(f"Double 5: {double(5)}")
print(f"Triple 5: {triple(5)}")

# A closure is a function + captured environment
print(f"\nClosure details:")
print(f"  __closure__: {double.__closure__}")
print(f"  Cell contents: {double.__closure__[0].cell_contents}")

# Practical closure — counter
def make_counter():
    """Return a counter function (closure captures count)."""
    count = 0
    def counter():
        nonlocal count  # needed to modify captured variable
        count += 1
        return count
    return counter

my_counter = make_counter()
print(f"Counter: {my_counter()}, {my_counter()}, {my_counter()}")

## Decorators — Fundamentals

In [ ]:
# A decorator is a function that takes a function and returns a function

def my_decorator(func):
    """Simple decorator that wraps a function."""
    def wrapper(*args, **kwargs):
        print(f"Before calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"After calling {func.__name__}")
        return result
    return wrapper

# Apply decorator manually (old way)
def say_hello(name):
    return f"Hello {name}!"

say_hello = my_decorator(say_hello)
print(say_hello("Alice"))

print("\n--- Using @ syntax ---")

# Using @ syntax (modern way)
@my_decorator
def say_goodbye(name):
    return f"Goodbye {name}!"

print(say_goodbye("Bob"))

## functools.wraps

Without `@wraps`, the decorated function loses its **metadata** (name, docstring, etc.).

In [ ]:
from functools import wraps

# Bad decorator — loses metadata
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        """Wrapper docstring."""
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def add(a, b):
    """Add two numbers."""
    return a + b

print(f"Bad decorator:")
print(f"  Name: {add.__name__}")      # 'wrapper'
print(f"  Doc:  {add.__doc__}")       # 'Wrapper docstring.'

# Good decorator — uses @wraps
def good_decorator(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        """Wrapper docstring."""
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def multiply(a, b):
    """Multiply two numbers."""
    return a * b

print(f"\nGood decorator:")
print(f"  Name: {multiply.__name__}")  # 'multiply'
print(f"  Doc:  {multiply.__doc__}")   # 'Multiply two numbers.'

## Common Decorator Patterns

In [ ]:
from functools import wraps
import time

# Pattern 1: Timing decorator
def timer(func):
    """Measure and print the execution time of a function."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f}s")
        return result
    return wrapper

@timer
def slow_function():
    """A function that sleeps for 0.5 seconds."""
    time.sleep(0.5)
    return "Done!"

print(slow_function())

In [ ]:
from functools import wraps

# Pattern 2: Debug decorator
def debug(func):
    """Print the function call details (name, args, kwargs, return value)."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        args_repr = [repr(a) for a in args]
        kwargs_repr = [f"{k}={v!r}" for k, v in kwargs.items()]
        signature = ", ".join(args_repr + kwargs_repr)
        print(f"Calling: {func.__name__}({signature})")
        result = func(*args, **kwargs)
        print(f"Returned: {func.__name__}({signature}) = {result!r}")
        return result
    return wrapper

@debug
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

greet("Alice")
print()
greet("Bob", greeting="Hi")

## Decorators with Arguments

Sometimes you want to pass arguments to a decorator itself. This requires a **triple-nested** function.

In [ ]:
from functools import wraps

# Decorator with arguments
def repeat(num_times):
    """Decorator that repeats the function call num_times times."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(num_times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(num_times=3)
def announce(msg):
    print(f"Announcement: {msg}")

announce("Meeting at 3pm!")

print("\n--- Demo: retry decorator ---")

# Retry decorator
def retry(max_attempts=3, delay=0.1):
    """Retry a function if it raises an exception."""
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    if attempt == max_attempts:
                        raise
                    print(f"Attempt {attempt} failed: {e}. Retrying...")
                    time.sleep(delay)
            return None
        return wrapper
    return decorator

import time

@retry(max_attempts=3)
def unstable_network_call():
    """Simulate an unreliable network call."""
    import random
    if random.random() < 0.7:  # 70% failure rate
        raise ConnectionError("Network timeout")
    return "Data received!"

result = unstable_network_call()
print(f"Result: {result}")

## Multiple Decorators (Stacking)

In [ ]:
from functools import wraps
import time

# Decorators
def bold(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<b>{func(*args, **kwargs)}</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return f"<i>{func(*args, **kwargs)}</i>"
    return wrapper

# Stacking decorators
# Order: bottom-up application, top-down execution
@bold
@italic
def format_text(text):
    return text

# Same as: format_text = bold(italic(format_text))
# Execution: bold's wrapper calls italic's wrapper calls format_text

print(format_text("Hello World"))
# Output: <b><i>Hello World</i></b>

## Class-Based Decorators

Decorators can also be implemented as classes using `__call__`.

In [ ]:
from functools import wraps

# Class-based decorator
class CountCalls:
    """Decorator that counts how many times a function is called."""
    def __init__(self, func):
        self.func = func
        self.count = 0
    
    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f"Call #{self.count} to {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def say_hi(name):
    return f"Hi, {name}!"

print(say_hi("Alice"))
print(say_hi("Bob"))
print(say_hi("Charlie"))
print(f"\nTotal calls: {say_hi.count}")

## Interview Problems

### Problem 1: Timing Decorator
Create a decorator that measures execution time and logs it.

In [ ]:
from functools import wraps
import time

def timing_decorator(func):
    """Measure and store execution time."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        wrapper.total_time += elapsed
        wrapper.call_count += 1
        wrapper.last_time = elapsed
        return result
    
    wrapper.total_time = 0
    wrapper.call_count = 0
    wrapper.last_time = 0
    return wrapper

@timing_decorator
def process_data(n):
    """Simulate data processing."""
    total = 0
    for i in range(n):
        total += i ** 2
    time.sleep(0.1)
    return total

# Test
process_data(1000)
process_data(2000)
process_data(3000)

print(f"Call count: {process_data.call_count}")
print(f"Total time: {process_data.total_time:.4f}s")
print(f"Last time:  {process_data.last_time:.4f}s")

### Problem 2: Memoization Decorator
Cache function results to avoid redundant computation.

In [ ]:
from functools import wraps

def memoize(func):
    """Cache function results based on arguments."""
    cache = {}
    
    @wraps(func)
    def wrapper(*args, **kwargs):
        # Create a hashable key from args and kwargs
        key = args + tuple(sorted(kwargs.items()))
        if key not in cache:
            cache[key] = func(*args, **kwargs)
        return cache[key]
    
    wrapper.cache = cache
    wrapper.cache_info = lambda: {"size": len(cache), "hits": wrapper.hits}
    wrapper.hits = 0
    return wrapper

@memoize
def fibonacci(n):
    """Recursive Fibonacci with memoization."""
    if n <= 1:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

# Test
import time
start = time.perf_counter()
result = fibonacci(35)
print(f"Fibonacci(35) = {result}")
print(f"Time: {time.perf_counter() - start:.4f}s")
print(f"Cache size: {len(fibonacci.cache)}")

# Without memoization, fibonacci(35) would take much longer
# Time: O(n) with memo, O(2^n) without
# Space: O(n)

### Problem 3: Login-Required Decorator
Simulate an authentication decorator for a web framework.

In [ ]:
from functools import wraps

# Simulated user session store
_sessions = {}

class User:
    def __init__(self, username, role="user"):
        self.username = username
        self.role = role

# Simulated request object
class Request:
    def __init__(self, session_token=None):
        self.session_token = session_token
        self.user = _sessions.get(session_token)

def login_required(func):
    """Decorator that requires a user to be logged in."""
    @wraps(func)
    def wrapper(request, *args, **kwargs):
        if not request.user:
            raise PermissionError("Login required")
        return func(request, *args, **kwargs)
    return wrapper

def admin_required(func):
    """Decorator that requires admin role."""
    @wraps(func)
    def wrapper(request, *args, **kwargs):
        if not request.user:
            raise PermissionError("Login required")
        if request.user.role != "admin":
            raise PermissionError("Admin access required")
        return func(request, *args, **kwargs)
    return wrapper

# Test
@login_required
def view_profile(request):
    return f"Profile of {request.user.username}"

@admin_required
def admin_panel(request):
    return "Admin Panel"

# Simulate a logged-in session
_sessions["abc123"] = User("Alice", role="admin")
req = Request("abc123")

print(f"Profile: {view_profile(req)}")
print(f"Admin:   {admin_panel(req)}")

# Simulate unauthenticated request
try:
    view_profile(Request())
except PermissionError as e:
    print(f"\nUnauthenticated error: {e}")

## Summary

| Concept | Description |
|---------|-------------|
| First-class functions | Functions as variables, arguments, return values |
| Closures | Nested function + captured environment |
| Decorator `@` | `@decorator` syntax sugar |
| `@wraps` | Preserves function metadata |
| Decorator args | Triple-nested functions |
| Stacking | Multiple `@` decorators |
| Class decorator | Using `__call__` protocol |

**Next**: Move to **02_Iterators_Generators_Advanced.ipynb**.